Clustering graph nodes to find emergent subsystems for higher level documentation

In [1]:
import json
from pathlib import Path
from typing import List, Dict, Set, Tuple
import networkx as nx
import sys
import os
from pprint import pprint


os.environ.pop("http_proxy", None)
os.environ.pop("https_proxy", None)
os.environ.pop("HTTP_PROXY", None)
os.environ.pop("HTTPS_PROXY", None)

# --- Config ---
PROJECT_ROOT = Path('../..').resolve()  # Adjust as needed to find the project root
GRAPH_PATH = PROJECT_ROOT / ".ai/context/internal/code_graph.gml"
DATABASE_PATH = PROJECT_ROOT / ".ai/context/internal/code_graph.json"
PRIORITY_PATH = PROJECT_ROOT / ".ai/context/internal/forward_pass_schedule.json"
DOX_DIR = PROJECT_ROOT / ".ai/context/internal/generated_dox"
XML_DIR = PROJECT_ROOT / ".ai/context/doxygen_output/xml"


# Add the modules directory to the Python path if needed
modules_dir = PROJECT_ROOT / '.ai/modules'
if str(modules_dir) not in sys.path:
    sys.path.append(str(modules_dir))

SRC_DIR = PROJECT_ROOT / 'src'
TEMP_DIR = PROJECT_ROOT / 'tmp'

print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/qte2333/repos/legacy


In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import doxygen_parse as dp

db = dp.load_db(DATABASE_PATH)

In [4]:
import doxygen_graph as dg

# --- Load graph and priority ---
full_graph: nx.DiGraph = dg.load_graph(GRAPH_PATH)
subgraph_nodes = [
    n for n, data in full_graph.nodes(data=True)
    if data.get('type') in {dg.EntityType.COMPOUND, dg.EntityType.MEMBER}
    and data.get('kind') not in {
        'dir', 'file', 'namespace'
    }
]
graph = full_graph.subgraph(subgraph_nodes).copy()
len(graph)

2025-06-22 09:07:01.219 | INFO     | doxygen_graph:load_graph:462 - Graph loaded from /Users/qte2333/repos/legacy/.ai/context/internal/code_graph.gml, nodes: 14517, edges: 50282


4549

In [5]:
# Find nodes in 'graph' with no incoming or outgoing edges
isolated_nodes = [n for n in graph.nodes if graph.degree(n) == 0]
print(f"Found {len(isolated_nodes)} isolated nodes:")
print(isolated_nodes)

# Remove isolated nodes from the graph
graph.remove_nodes_from(isolated_nodes)
print(f"Graph now has {graph.number_of_nodes()} nodes and {graph.number_of_edges()} edges after removal.")


Found 3 isolated nodes:
['1a548f0c2ae589c8ea4a1bab43d198854e', '1aec8f2c2bff6245a491ae4d2582d58afb', '1a3e23a54bc9aa6860fe74b6e882df5ef2']
Graph now has 4546 nodes and 36002 edges after removal.


In [6]:
# Find disconnected groups (weakly connected components) in the graph
disconnected_groups = list(nx.weakly_connected_components(graph))
# Exclude the largest group and print members of each smaller group
print(f"Found {len(disconnected_groups)} disconnected groups.")
largest_group = max(disconnected_groups, key=len)
print(f"Large group size: {len(largest_group)}; Sample nodes: {list(largest_group)[:5]}")
smaller_groups = [g for g in disconnected_groups if g is not largest_group]

print("\nMembers of each smaller disconnected group:")
for idx, group in enumerate(smaller_groups):
    print(f"Group {idx} (size {len(group)}): {list(group)}")

graph = graph.subgraph(largest_group)

Found 12 disconnected groups.
Large group size: 4529; Sample nodes: ['1aacdd5193fc5d4771523b0fd62075cc2f', '1aced03971d13897c52daaf024b0efebc1', '1ga08335f40a1d6e78d1e3f534abef2afc4', 'class_world', '1a17a3a6653e3a7e583b4cc7fa956cd96c']

Members of each smaller disconnected group:
Group 0 (size 5): ['1ab5186b25ff0f451cc8627d92e0c6e47d', '1ad5b64f096f955c359444bbeb6b20d97a', '1ab03206012172898f79c49fa0439931d9', 'structgem_1_1quality__st', '1a5235e14a57b4ce6f1ef7ecc7e1ef7048']
Group 1 (size 1): ['1a79bcc699f0e754ac8f4259a2c1299525']
Group 2 (size 1): ['1a867447dda0be0c8123d273bd6f94747a']
Group 3 (size 3): ['1a12ad273e99a1dcf3411abcbfb96f51fc', '1a65f116fe0c18b66cbd441118185896da', 'structprd__table__entry']
Group 4 (size 1): ['1adff9323a34317c4953c3dd17d49eff9d']
Group 5 (size 1): ['1a70fc1fe76afb606fc36c41ad9a9f3c38']
Group 6 (size 1): ['1add44836f0587a5b62f1b1d838def2531']
Group 7 (size 1): ['1af9bd51329e4672f609e13e73138e8a81']
Group 8 (size 1): ['1ab288c9b2d74f69214e9397c8077c352d'

In [7]:
import doc_db

In [9]:
import networkx as nx
from collections import Counter
import math

def identify_utility_nodes(graph: nx.DiGraph, out_degree_threshold=20, centrality_threshold=0.05):
    """
    Identify utility-like nodes in a call/dependency graph.

    Args:
        graph (nx.DiGraph): The input graph.
        out_degree_threshold (int): Minimum out-degree to consider a node utility-like.
        centrality_threshold (float): Minimum betweenness centrality to consider a node utility-like.

    Returns:
        Set of node IDs that are likely utility nodes.
    """
    nodes = {n:{} for n in graph}
    for n, d in graph.out_degree():
        nodes[n]["out_degree"] = d
    for n, d in graph.in_degree():
        nodes[n]["in_degree"] = d

    for n, c in betweenness.items():
        nodes[n]["betweenness"] = c

    max_in_degree = max(d["in_degree"] for d in nodes.values())
    max_out_degree = max(d["out_degree"] for d in nodes.values())
    max_betweenness = max(d["betweenness"] for d in nodes.values())
    print(max_betweenness, max_out_degree, max_in_degree)
    normalized = {
        n: {
            "out_degree": d["out_degree"] / max_out_degree if max_out_degree > 0 else 0,
            "in_degree": d["in_degree"] / max_in_degree if max_in_degree > 0 else 0,
            "betweenness": d["betweenness"] / max_betweenness if max_betweenness > 0 else 0
        }
        for n, d in nodes.items()
    }

    ordered = sorted(normalized.items(), key=lambda x: x[1]["out_degree"] * (1 - x[1]['in_degree']) * x[1]["betweenness"], reverse=True)
    return ordered


In [34]:
# Use Leiden clustering to identify communities in the graph
import igraph as ig
from leidenalg import find_partition, RBConfigurationVertexPartition

# Convert the NetworkX graph to an iGraph graph
def convert_to_igraph(nx_graph: nx.DiGraph) -> ig.Graph:
    ig_graph = ig.Graph(directed=True)
    ig_graph.add_vertices(list(nx_graph.nodes))
    ig_graph.add_edges(list(nx_graph.edges))
    return ig_graph

# Perform Leiden clustering
def perform_leiden_clustering(graph: nx.DiGraph) -> Dict[int, List[str]]:
    ig_graph = convert_to_igraph(graph)
    partition = find_partition(ig_graph, n_iterations=-1, partition_type=RBConfigurationVertexPartition)
    
    # Map communities to node IDs
    communities = {}
    for node, community in enumerate(partition.membership):
        communities.setdefault(community, []).append(ig_graph.vs[node]["name"])
    return communities

N = 50
cluster_runs = []

for _ in range(N):
    communities = perform_leiden_clustering(graph)
    cluster_runs.append([set(nodes) for nodes in communities.values()])

# supergroups = []

# for _ in range(N):
#     # Apply clustering to the graph
#     communities = perform_leiden_clustering(graph)
#     for index, nodes in communities.items():
#         found_match = False
#         for group in supergroups:
#             different = False
#             for node in nodes:
#                 if node in group:
#                     group[node] += 1
#                 else:
#                     different = True
#             if not different:
#                 found_match = True
#         if not found_match:
#             supergroups.append({n:1 for n in nodes})

# core_groups: List[Set[str]] = []
# for group in supergroups:
#     frequency = max(group.values())
#     core_group = set(n for n in group if group[n] == frequency)
#     core_groups.append(core_group)

# core_groups = {cg for cg in core_groups if not any(cg < other for other in core_groups if cg != other)}
# print(sum(len(group) for group in core_groups))
# for group in core_groups:
#     print(f"Core group: {len(group)} {group}")

In [37]:
from collections import defaultdict
from itertools import combinations

def find_core_groups(cluster_runs, threshold=0.8):
    """
    Identify core groups by tracking pairwise co-occurrence.
    """
    pair_counts = defaultdict(int)
    total_runs = len(cluster_runs)

    for run in cluster_runs:
        for cluster in run:
            for a, b in combinations(sorted(cluster), 2):
                pair_counts[(a, b)] += 1

    # Build co-occurrence graph
    import networkx as nx
    G = nx.Graph()
    for (a, b), count in pair_counts.items():
        # if count == total_runs:
        if count / total_runs >= threshold:
            G.add_edge(a, b)

    return [set(c) for c in nx.connected_components(G)]

core_groups = find_core_groups(cluster_runs, threshold=0.9)

print(sum(len(group) for group in core_groups))
print(len(core_groups))
for group in core_groups:
    print(f"Core group: {len(group)} {group}")


4261
149
Core group: 254 {'1ade555e7304db34e8e6cba6dff89fcfa4', '1aacdd5193fc5d4771523b0fd62075cc2f', '1a94e3d10978530596db5f5a8f36df123e', '1a5906faf273291116b34822c9acc3b55c', '1a646a9e344562f1df63e6cdadc961fa9c', '1a934d2fd5bfb002e27e70452da018e47c', 'class_world', '1ga22b55031a4cf5dc015c94205fc3d8751', '1a4335edd4861561d2192d6a5f0feea370', '1a4a37d22982dde939f75e7f9d3f3db967', '1a5383cc287fed9744b75064d3c2a926bf', '1a4ade2a341e71b692dd350df067d8a8bb', '1af653b70dae82953f5a458614ebbc31b6', '1ae79bd5242948154d49cc97c92bb119c8', '1ac220120693ae90bd7b2421a8bdf44511', '1a3c0c77b4c2b07850300ea12f86b59e9a', '1a82247d3145f8e05b258a3bf6cf2112d9', '1ab4df38e8c521e7aeef9589ccb637d2ca', '1ab4c9d15b695f0a60f9b7059c6731e7d9', '1a3d001ad1daa359110d4ea7fa06f2b0a7', '1abeeb5e70077df9861ed67b6695ab5f26', '1a41bfe943da201cae2ec289665396871c', '1a002ac6c3207be1c12fea9fc5f1e19818', '1adec84a235443abc23f50b58bd1c846e7', '1a5e68dfadf99183a00db6b793e5275c13', '1a8c73fba541a5817fff65147ba47cd827', '1a26545

In [ ]:
def measure_intergroup_connectivity(graph, groups):
    """
    Computes pairwise edge connectivity between groups.

    Args:
        graph: A NetworkX graph.
        groups: List of sets of node names (core groups).

    Returns:
        A list of tuples: (i, j, edge_count), where i < j and i, j are group indices.
    """
    connectivity = []
    for i in range(len(groups)):
        for j in range(i + 1, len(groups)):
            edges = 0
            for u in groups[i]:
                for v in groups[j]:
                    if graph.has_edge(u, v):
                        edges += 1
            connectivity.append((i, j, edges))
    return connectivity

def group_adjacency_graph(connectivity, threshold):
    """
    Build a graph where each node is a group index, and edges exist if connectivity exceeds threshold.
    """
    import networkx as nx
    G = nx.Graph()
    for i, j, count in connectivity:
        if count >= threshold:
            G.add_edge(i, j)
    return G

def merge_connected_groups(groups, group_graph):
    """
    Merge groups based on connected components of the group-graph.
    """
    merged = []
    for component in nx.connected_components(group_graph):
        merged_group = set()
        for idx in component:
            merged_group.update(groups[idx])
        merged.append(merged_group)
    return merged

connectivity = measure_intergroup_connectivity(graph, core_groups)
group_graph = group_adjacency_graph(connectivity, threshold=5)  # Adjust threshold as needed
merged_groups = merge_connected_groups(core_groups, group_graph)

In [ ]:

# Print the identified communities
print(f"Identified {len(communities)} communities:")
for community_id, nodes in communities.items():
    print(f"Community {community_id}: {len(nodes)} nodes")
    print(nodes[:10])  # Print the first 10 nodes in each community

# Write the community lists of nodes to a file
output_file = PROJECT_ROOT / ".ai/context/internal/leiden_communities.json"
with output_file.open("w", encoding="utf-8") as f:
    json.dump(communities, f, indent=2)
print(f"Community node lists written to {output_file}")

In [19]:
from jinja2 import Environment, FileSystemLoader
from llm_utils import call_openai, OPENAI_MODEL, OPENAI_MAX_TOKENS
import json

# Shared environment setup for Jinja2
template_dir = PROJECT_ROOT / ".ai/templates"
env = Environment(loader=FileSystemLoader(template_dir))

def render_template(template_name: str, **context) -> str:
    template = env.get_template(template_name)
    return template.render(**context)

def classify_community(community_id: int, community_nodes: List[str], graph: nx.DiGraph) -> Dict:
    """
    Classify a community of nodes using an LLM and the provided Jinja2 template.

    Args:
        community_id: The ID of the community.
        community_nodes: List of node IDs in the community.
        graph: The graph containing the nodes.

    Returns:
        A dictionary containing the classification result.
    """
    # Extract entity summaries from the graph
    entities = []
    for node in community_nodes:
        eid = dg.get_body_eid(db, node)
        e = db.get(eid)
        if e.kind in ('variable',):
            continue
        doc = doc_db.get_doc(eid.compound, e.signature)
        entities.append({
            "name": doc.name,
            "brief": doc.brief or "No summary available."
        })

    # Render the prompt using the Jinja2 template
    prompt = render_template("docgen_cluster_candidate.j2", entities=entities)
    print(prompt)

    # Query the LLM
    response_text = call_openai(prompt, OPENAI_MODEL, OPENAI_MAX_TOKENS)

    # Parse the response
    try:
        import re
        response_text = response_text.lstrip("```json").rstrip("```")
        # Add an extra backslash to single backslashes not part of an escaped quote or escaped single quote
        response_text = re.sub(r'\\(?!["\'])', r'\\\\', response_text)
        doc_json = json.loads(response_text)
        pprint(doc_json)
        return {k: v for k, v in doc_json.items() if v}
    except json.JSONDecodeError as e:
        print(f"Failed to parse LLM response for community {community_id}: {e}")
        print(f"Response text: {response_text}")
        return None

# Iterate over communities from smallest to largest
community_sizes = {cid: len(nodes) for cid, nodes in communities.items()}
sorted_communities = sorted(communities.items(), key=lambda x: len(x[1]))

results = {}
for community_id, community_nodes in sorted_communities:
    print(f"Classifying community {community_id} with {len(community_nodes)} nodes...")
    result = classify_community(community_id, community_nodes, graph)
    if result:
        results[community_id] = result

# Save the classification results to a JSON file
output_path = PROJECT_ROOT / ".ai/context/internal/community_classifications.json"
with output_path.open("w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

print(f"Community classifications saved to {output_path}")

Classifying community 37 with 4 nodes...
You are evaluating whether a set of C++ entities forms a coherent subsystem.

Each entity has a summary describing its purpose or behavior. These summaries were generated from source code and usage context. Your job is to analyze whether the grouping seems functionally and semantically appropriate.

Please answer the following questions:
- Do these entities appear to work together as part of a unified concept or purpose?
- If so, what is the likely purpose or functionality of this subsystem?
- Are there any entities that seem out of place, overly generic, or belong elsewhere?
- If applicable, suggest a concise name or label for this group.

Respond in this structured format:

```json
{
  "coherent": true | false,
  "subsystem_name": "concise descriptive label, if coherent",
  "summary": "one-paragraph explanation of the subsystem's role or purpose",
  "outliers": ["list", "of", "entity", "names", "if", "any", "seem", "misplaced"]
}
```

Here is 

In [18]:
for node in communities[2]:
    print(node, graph.nodes[node].get("name", node))

1ace418ff58211455133a0888a841202f7 void spell_cure_blindness(skill::type sn, int level, Character *ch, void *vo, int target, int evolution)
1aa08ec13f6f08a5a326a48c72f1fdaca3 const skill_table_t & skill::lookup(type t)
1a1371b12d48778edf2f829db4ee1d6fbb void do_scribe(Character *ch, String argument)
1afb88ac3abdff644b423cfa13fc12c179 int group_gain(Character *ch, Character *victim)
1gad2b0ae32ff6929b06025078fcee55254 ROOM_NO_RECALL
1a9b927885ded1744afaff4cff3c0206b6 IS_GOOD
1a2477890b9ef5c98e97920eaa65d6c7ea void do_recall(Character *ch, String argument)
1a06434da0476248a1d464bc5aa5477d9a void spell_sleep(skill::type sn, int level, Character *ch, void *vo, int target, int evolution)
1a4756179d0947f7ad6314a46f0e8c04e0 void spell_detect_magic(skill::type sn, int level, Character *ch, void *vo, int target, int evolution)
1a7ad8db71d07508d1f68a44dea1564cc6 void combat_regen(Character *ch)
1aec91ff0d453fb734225561fc33347ea2 void do_shadow(Character *ch, String argument)
1a604ed1b43660edc714

In [17]:
def recursive_leiden(graph: nx.DiGraph, community_nodes: List[str], min_size=10, max_size=150):
    subgraph = graph.subgraph(community_nodes)
    if subgraph.number_of_nodes() < min_size:
        return [community_nodes]

    refined = perform_leiden_clustering(subgraph)
    # Optionally recurse again if still too large
    result = []
    for cluster in refined.values():
        if len(cluster) > max_size:
            result.extend(recursive_leiden(graph, cluster))
        else:
            result.append(cluster)
    return result

large_communities = {k: v for k, v in communities.items() if len(v) > 150}
refined = []
for community_id, nodes in large_communities.items():
    refined += recursive_leiden(graph, nodes)
print(f"Refined into {len(refined)} smaller subcommunities")

Refined into 114 smaller subcommunities


In [ ]:
import networkx as nx
from typing import Set, Dict, List, Tuple, Optional
import warnings

def identify_system_clusters(
    graph: nx.Graph,
    utility_nodes: Optional[Set[str]] = None,
    use_leiden: bool = True,
    weight: Optional[str] = None,
    resolution: float = 1.0
) -> List[Set[str]]:
    """
    Identify clusters (systems) of nodes in a graph using Leiden or Louvain community detection.

    Args:
        graph: The full dependency graph (undirected or directed).
        utility_nodes: Optional set of utility node IDs to exclude from influencing clustering.
        use_leiden: If True, uses the Leiden algorithm (requires backend); otherwise, Louvain.
        weight: Optional edge attribute to use as weight.
        resolution: Resolution parameter for the community detection algorithm.

    Returns:
        A list of sets, where each set contains node IDs that form a system cluster.
    """
    try:
        # Clone and preprocess the graph
        work_graph = graph.copy()
        utility_nodes = utility_nodes or set()

        # Exclude utility nodes by removing them temporarily
        clustering_graph = work_graph.subgraph(
            [n for n in work_graph.nodes if n not in utility_nodes]
        ).copy()

        # Ensure the graph is undirected for community detection
        if graph.is_directed():
            clustering_graph = clustering_graph.to_undirected()

        # Run community detection
        perform_leiden_clustering(graph)
        if use_leiden:
            communities = leiden_communities(clustering_graph, weight=weight, resolution_parameter=resolution)
        else:
            communities = louvain_communities(clustering_graph, weight=weight, resolution=resolution)

        # Return each community as a set of node IDs
        return [set(community) for community in communities]

    except ImportError as e:
        warnings.warn("Leiden algorithm requires the 'leidenalg' and 'python-igraph' packages. Falling back to Louvain.")
        communities = louvain_communities(clustering_graph, weight=weight, resolution=resolution)
        return [set(community) for community in communities]

identify_system_clusters(graph)

NotImplementedError: 'leiden_communities' is not implemented by 'networkx' backend. This function is included in NetworkX as an API to dispatch to other backends.

In [10]:
# Save the communities to a JSON file for further analysis
output_path = TEMP_DIR / "graph_communities.json"

with output_path.open("w", encoding="utf-8") as f:
    json.dump(communities, f, indent=2)

print(f"Communities saved to {output_path}")

Communities saved to /Users/qte2333/repos/legacy/tmp/graph_communities.json


In [ ]:
# Visualize the communities using a force-directed layout
import matplotlib.pyplot as plt

def visualize_communities(graph: nx.DiGraph, communities: Dict[int, List[str]]):
    pos = nx.spring_layout(graph, seed=42)  # Force-directed layout
    plt.figure(figsize=(12, 12))
    
    # Assign colors to communities
    community_colors = {community: plt.cm.tab20(i % 20) for i, community in enumerate(communities.keys())}
    
    for community_id, nodes in communities.items():
        nx.draw_networkx_nodes(
            graph, pos, nodelist=nodes, node_color=[community_colors[community_id]],
            label=f"Community {community_id}", node_size=50
        )
    
    nx.draw_networkx_edges(graph, pos, alpha=0.3)
    plt.legend()
    plt.title("Graph Communities Identified by Leiden Clustering")
    plt.show()

visualize_communities(graph, communities)